# MVSL-DSF with UNIFIED SPLITS
This notebook provides the full training loop for MVSL-DSF using Split A (warm-start) and Split B (cold-start).

**Important Caveat for Cold-start (Split B):**
MVSL-DSF uses Views 4-5 (drug association and frequency profiles) as inputs. In a true cold-start scenario, test drugs have no known associations, making these profiles empty. If we do not zero out these rows for test drugs during training, there will be data leakage. We must strictly zero out the test drug rows in the profile matrices to simulate this cold-start setting accurately.


In [ ]:
# ===== SETUP =====
REPO_URL = 'https://github.com/hungbuile04/DSE.git'

import os
os.chdir('/content')
!git clone $REPO_URL

!pip install -q torch dgl dgllife rdkit-pypi scipy scikit-learn numpy


In [ ]:
# ===== DATA PREPARATION =====
import os
import shutil

src_dir = '/content/DSE/shared_data/mvsl_750_994'
dst_dir = '/content/DSE/MVSL-DSF'
os.makedirs(dst_dir, exist_ok=True)
for f in os.listdir(src_dir):
    shutil.copy2(os.path.join(src_dir, f), os.path.join(dst_dir, f))
    print(f'Copied: {f}')


In [ ]:
# ===== PRECOMPUTE SMILES FEATURES =====
import sys
sys.path.append('/content/DSE')
sys.path.append('/content/DSE/shared_data')
sys.path.insert(0, '/content/DSE/MVSL-DSF')

import pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, mean_squared_error, mean_absolute_error
from rdkit import Chem
from rdkit.Chem import AllChem
import dgl
from dgllife.utils import smiles_to_bigraph, AttentiveFPAtomFeaturizer, AttentiveFPBondFeaturizer

from split_adapter import load_splits, load_test_drugs, make_masked_freq
from Network import ConvNCF

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATA_DIR = '/content/DSE/MVSL-DSF'
with open(f'{DATA_DIR}/drug_side_association_matrix.pkl', 'rb') as f:
    drug_side_assoc = pickle.load(f)
with open(f'{DATA_DIR}/drug_side_frequency_matrix.pkl', 'rb') as f:
    drug_side_freq = pickle.load(f)
with open(f'{DATA_DIR}/side_vector_level_123.pkl', 'rb') as f:
    side_vector = pickle.load(f)
with open(f'{DATA_DIR}/drug_smiles.pkl', 'rb') as f:
    drug_smiles_data = pickle.load(f)
with open(f'{DATA_DIR}/final_sample.pkl', 'rb') as f:
    final_sample = pickle.load(f)

SMILES_CHARS = ['?','#','%',')','(','+','-','.','1','0','3','2','5','4',
                '7','6','9','8','=','A','C','B','E','D','G','F','I',
                'H','K','M','L','O','N','P','S','R','U','T','W','V',
                'Y','[','Z',']','_','a','c','b','e','d','g','f','i',
                'h','m','l','o','n','s','r','u','t','y']
MAX_SEQ = 100
CHAR_DIM = 63

def smiles_to_onehot(smi):
    smi = smi[:MAX_SEQ].ljust(MAX_SEQ, '?')
    mat = np.zeros((MAX_SEQ, CHAR_DIM), dtype=np.float32)
    for i, c in enumerate(smi):
        idx = SMILES_CHARS.index(c) if c in SMILES_CHARS else -1
        if idx >= 0: mat[i, idx] = 1.0
    return mat

def smiles_to_fp(smi, nbits=2048):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return np.zeros(nbits, dtype=np.float32)
    return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=nbits), dtype=np.float32)

atom_featurizer = AttentiveFPAtomFeaturizer()
bond_featurizer = AttentiveFPBondFeaturizer(self_loop=True)

def smiles_to_graph(smi):
    g = smiles_to_bigraph(smi, add_self_loop=True,
                             node_featurizer=atom_featurizer,
                             edge_featurizer=bond_featurizer)
    if g is None:
        g = dgl.graph(([0], [0]))
        g.ndata['h'] = torch.zeros(1, atom_featurizer.feat_size())
        g.edata['e'] = torch.zeros(1, bond_featurizer.feat_size())
    return g

smiles_onehot_list, smiles_fp_list, smiles_graph_list = [], [], []
for item in sorted(drug_smiles_data, key=lambda x: x[0]):
    smi = item[1]
    smiles_onehot_list.append(smiles_to_onehot(smi))
    smiles_fp_list.append(smiles_to_fp(smi))
    smiles_graph_list.append(smiles_to_graph(smi))

smiles_onehot_array = np.array(smiles_onehot_list)
smiles_fp_array = np.array(smiles_fp_list)
side_vector_array = np.array(side_vector, dtype=np.float32)

base_config_TextCNN = {
    'dropout_rate': 0.3,
    'embedding_size': 64,
    'feature_size': 64,
    'max_text_len': MAX_SEQ,
    'window_sizes': [1],
}

class MVSLDataset(Dataset):
    def __init__(self, samples, assoc_matrix, freq_matrix):
        self.samples = samples
        self.assoc_matrix = assoc_matrix
        self.freq_matrix = freq_matrix
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        d, s, a, f = self.samples[idx]
        assoc_prof = self.assoc_matrix[d, :]
        freq_prof = self.freq_matrix[d, :]
        assoc_prof_T = self.assoc_matrix[:, s]
        return d, s, a, f, assoc_prof, freq_prof, assoc_prof_T

def collate_fn(batch):
    d_idx = [item[0] for item in batch]
    s_idx = [item[1] for item in batch]
    a_label = torch.LongTensor([item[2] for item in batch])
    f_label = torch.FloatTensor([item[3] / 5.0 for item in batch])
    
    assoc_prof = torch.FloatTensor(np.array([item[4] for item in batch]))
    freq_prof = torch.FloatTensor(np.array([item[5] for item in batch]))
    assoc_prof_T = torch.FloatTensor(np.array([item[6] for item in batch]))
    
    graphs = [smiles_graph_list[i] for i in d_idx]
    onehots = torch.FloatTensor(smiles_onehot_array[d_idx])
    fps = torch.FloatTensor(smiles_fp_array[d_idx])
    side_vec = torch.FloatTensor(side_vector_array[s_idx])
    
    return d_idx, s_idx, a_label, f_label, assoc_prof, freq_prof, assoc_prof_T, graphs, onehots, fps, side_vec


## Split A (Warm-start)


In [ ]:
%%time
NUM_EPOCHS = 50
BATCH_SIZE = 128
LR = 0.0001
DROPOUT = 0.3
HID_DIM = 64

os.makedirs('/content/DSE/results/MVSL-DSF_A', exist_ok=True)

for fold in range(10):
    print(f"=== Split A - Fold {fold} ===")
    train_pos, test_pos = load_splits(fold, 'A')
    
    masked_freq = make_masked_freq(drug_side_freq, test_pos)
    masked_assoc = (masked_freq > 0).astype(float)
    
    train_samples = []
    for r in train_pos:
        train_samples.append([int(r[0]), int(r[1]), 1, r[2]])
        
    test_set = set(map(tuple, test_pos[:, :2].astype(int)))
    neg_indices = np.argwhere(masked_freq == 0)
    neg_indices = np.array([idx for idx in neg_indices if tuple(idx) not in test_set])
    np.random.shuffle(neg_indices)
    neg_train = neg_indices[:len(train_pos)]
    for r in neg_train:
        train_samples.append([int(r[0]), int(r[1]), 0, 0.0])
        
    test_samples = []
    for r in test_pos:
        test_samples.append([int(r[0]), int(r[1]), 1, r[2]])
        
    neg_test = neg_indices[len(train_pos):len(train_pos)+len(test_pos)]
    for r in neg_test:
        test_samples.append([int(r[0]), int(r[1]), 0, 0.0])
        
    # Data leakage fix for Views 4-5 in warm-start
    assoc_train = drug_side_assoc.copy()
    freq_train = drug_side_freq.copy()
    for r in test_pos:
        assoc_train[int(r[0]), int(r[1])] = 0
        freq_train[int(r[0]), int(r[1])] = 0
        
    train_dataset = MVSLDataset(train_samples, assoc_train, freq_train)
    test_dataset = MVSLDataset(test_samples, assoc_train, freq_train)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    
    model = ConvNCF(ad_dim=[994, 750], drug_embed_dim=2048, 
                    side_embed_dim=[243], hid_embed_dim=HID_DIM, dropout=DROPOUT,
                    **base_config_TextCNN).to(DEVICE)
                    
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion_cls = nn.CrossEntropyLoss()
    criterion_freq = nn.BCEWithLogitsLoss()
    
    for epoch in range(NUM_EPOCHS):
        model.train()
        for batch in train_loader:
            d_idx, s_idx, a_label, f_label, assoc_prof, freq_prof, assoc_prof_T, graphs, onehots, fps, side_vec = batch
            a_label = a_label.to(DEVICE)
            f_label = f_label.to(DEVICE)
            assoc_prof = assoc_prof.to(DEVICE)
            freq_prof = freq_prof.to(DEVICE)
            assoc_prof_T = assoc_prof_T.to(DEVICE)
            onehots = onehots.to(DEVICE)
            fps = fps.to(DEVICE)
            side_vec = side_vec.to(DEVICE)
            
            optimizer.zero_grad()
            cls_out, freq_out = model(graphs, onehots, fps, None, side_vec, None, 
                                      [assoc_prof, freq_prof], [assoc_prof_T], DEVICE)
                                      
            loss_cls = criterion_cls(cls_out, a_label)
            pos_mask = a_label == 1
            loss_freq = criterion_freq(freq_out[pos_mask], f_label[pos_mask]) if pos_mask.sum() > 0 else 0
            loss = loss_cls + loss_freq + model.consistency_loss
            
            loss.backward()
            optimizer.step()
            
    model.eval()
    all_a_labels, all_a_preds = [], []
    all_f_labels, all_f_preds = [], []
    
    with torch.no_grad():
        for batch in test_loader:
            d_idx, s_idx, a_label, f_label, assoc_prof, freq_prof, assoc_prof_T, graphs, onehots, fps, side_vec = batch
            a_label = a_label.to(DEVICE)
            assoc_prof = assoc_prof.to(DEVICE)
            freq_prof = freq_prof.to(DEVICE)
            assoc_prof_T = assoc_prof_T.to(DEVICE)
            onehots = onehots.to(DEVICE)
            fps = fps.to(DEVICE)
            side_vec = side_vec.to(DEVICE)
            
            cls_out, freq_out = model(graphs, onehots, fps, None, side_vec, None, 
                                      [assoc_prof, freq_prof], [assoc_prof_T], DEVICE)
            
            preds_cls = torch.softmax(cls_out, dim=1)[:, 1].cpu().numpy()
            preds_freq = torch.sigmoid(freq_out).cpu().numpy() * 5.0
            
            all_a_labels.extend(a_label.cpu().numpy())
            all_a_preds.extend(preds_cls)
            
            pos_mask = a_label.cpu().numpy() == 1
            all_f_labels.extend(f_label.numpy()[pos_mask] * 5.0)
            all_f_preds.extend(preds_freq[pos_mask])
            
    auc = roc_auc_score(all_a_labels, all_a_preds)
    aupr = average_precision_score(all_a_labels, all_a_preds)
    rmse = np.sqrt(mean_squared_error(all_f_labels, all_f_preds))
    mae = mean_absolute_error(all_f_labels, all_f_preds)
    print(f"Split A Fold {fold} - AUC: {auc:.4f}, AUPR: {aupr:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}")
    
    np.save(f'/content/DSE/results/MVSL-DSF_A/fold_{fold}_labels.npy', all_f_labels)
    np.save(f'/content/DSE/results/MVSL-DSF_A/fold_{fold}_preds.npy', all_f_preds)

## Split B (Cold-start)
**Important Caveat:**
- Views 1-3 (graph, SMILES, fingerprint) are drug-intrinsic and usable.
- Views 6-7 (SE features) are SE-intrinsic and usable.
- Views 4-5 (association/frequency profile) **MUST** have ALL test drug rows zeroed out to prevent data leakage in a cold-start setting.


In [ ]:
%%time
os.makedirs('/content/DSE/results/MVSL-DSF_B', exist_ok=True)

for fold in range(10):
    print(f"=== Split B - Fold {fold} ===")
    train_pos, test_pos = load_splits(fold, 'B')
    test_drug_indices = load_test_drugs(fold, 'B')
    
    masked_freq = make_masked_freq(drug_side_freq, test_pos)
    masked_assoc = (masked_freq > 0).astype(float)
    
    # Zero out ALL rows for test drugs in profile matrices
    assoc_train = drug_side_assoc.copy()
    freq_train = drug_side_freq.copy()
    for d in test_drug_indices:
        assoc_train[d, :] = 0
        freq_train[d, :] = 0
        
    train_samples = []
    for r in train_pos:
        train_samples.append([int(r[0]), int(r[1]), 1, r[2]])
        
    test_set = set(map(tuple, test_pos[:, :2].astype(int)))
    
    valid_neg_mask = (masked_freq == 0)
    for d in test_drug_indices:
        valid_neg_mask[d, :] = False
        
    neg_indices = np.argwhere(valid_neg_mask)
    neg_indices = np.array([idx for idx in neg_indices if tuple(idx) not in test_set])
    np.random.shuffle(neg_indices)
    neg_train = neg_indices[:len(train_pos)]
    for r in neg_train:
        train_samples.append([int(r[0]), int(r[1]), 0, 0.0])
        
    test_samples = []
    for r in test_pos:
        test_samples.append([int(r[0]), int(r[1]), 1, r[2]])
        
    test_neg_mask = (masked_freq == 0)
    train_drugs = list(set(range(assoc_train.shape[0])) - set(test_drug_indices))
    for d in train_drugs:
        test_neg_mask[d, :] = False
        
    neg_test_indices = np.argwhere(test_neg_mask)
    neg_test_indices = np.array([idx for idx in neg_test_indices if tuple(idx) not in test_set])
    np.random.shuffle(neg_test_indices)
    neg_test = neg_test_indices[:len(test_pos)]
    for r in neg_test:
        test_samples.append([int(r[0]), int(r[1]), 0, 0.0])
        
    train_dataset = MVSLDataset(train_samples, assoc_train, freq_train)
    test_dataset = MVSLDataset(test_samples, assoc_train, freq_train)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    
    model = ConvNCF(ad_dim=[994, 750], drug_embed_dim=2048, 
                    side_embed_dim=[243], hid_embed_dim=HID_DIM, dropout=DROPOUT,
                    **base_config_TextCNN).to(DEVICE)
                    
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion_cls = nn.CrossEntropyLoss()
    criterion_freq = nn.BCEWithLogitsLoss()
    
    for epoch in range(NUM_EPOCHS):
        model.train()
        for batch in train_loader:
            d_idx, s_idx, a_label, f_label, assoc_prof, freq_prof, assoc_prof_T, graphs, onehots, fps, side_vec = batch
            a_label = a_label.to(DEVICE)
            f_label = f_label.to(DEVICE)
            assoc_prof = assoc_prof.to(DEVICE)
            freq_prof = freq_prof.to(DEVICE)
            assoc_prof_T = assoc_prof_T.to(DEVICE)
            onehots = onehots.to(DEVICE)
            fps = fps.to(DEVICE)
            side_vec = side_vec.to(DEVICE)
            
            optimizer.zero_grad()
            cls_out, freq_out = model(graphs, onehots, fps, None, side_vec, None, 
                                      [assoc_prof, freq_prof], [assoc_prof_T], DEVICE)
                                      
            loss_cls = criterion_cls(cls_out, a_label)
            pos_mask = a_label == 1
            loss_freq = criterion_freq(freq_out[pos_mask], f_label[pos_mask]) if pos_mask.sum() > 0 else 0
            loss = loss_cls + loss_freq + model.consistency_loss
            
            loss.backward()
            optimizer.step()
            
    model.eval()
    all_a_labels, all_a_preds = [], []
    all_f_labels, all_f_preds = [], []
    
    with torch.no_grad():
        for batch in test_loader:
            d_idx, s_idx, a_label, f_label, assoc_prof, freq_prof, assoc_prof_T, graphs, onehots, fps, side_vec = batch
            a_label = a_label.to(DEVICE)
            assoc_prof = assoc_prof.to(DEVICE)
            freq_prof = freq_prof.to(DEVICE)
            assoc_prof_T = assoc_prof_T.to(DEVICE)
            onehots = onehots.to(DEVICE)
            fps = fps.to(DEVICE)
            side_vec = side_vec.to(DEVICE)
            
            cls_out, freq_out = model(graphs, onehots, fps, None, side_vec, None, 
                                      [assoc_prof, freq_prof], [assoc_prof_T], DEVICE)
            
            preds_cls = torch.softmax(cls_out, dim=1)[:, 1].cpu().numpy()
            preds_freq = torch.sigmoid(freq_out).cpu().numpy() * 5.0
            
            all_a_labels.extend(a_label.cpu().numpy())
            all_a_preds.extend(preds_cls)
            
            pos_mask = a_label.cpu().numpy() == 1
            all_f_labels.extend(f_label.numpy()[pos_mask] * 5.0)
            all_f_preds.extend(preds_freq[pos_mask])
            
    auc = roc_auc_score(all_a_labels, all_a_preds)
    aupr = average_precision_score(all_a_labels, all_a_preds)
    rmse = np.sqrt(mean_squared_error(all_f_labels, all_f_preds))
    mae = mean_absolute_error(all_f_labels, all_f_preds)
    print(f"Split B Fold {fold} - AUC: {auc:.4f}, AUPR: {aupr:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}")
    
    np.save(f'/content/DSE/results/MVSL-DSF_B/fold_{fold}_labels.npy', all_f_labels)
    np.save(f'/content/DSE/results/MVSL-DSF_B/fold_{fold}_preds.npy', all_f_preds)


## Results


In [ ]:
print("Training and evaluation complete. Results saved in /content/DSE/results/MVSL-DSF_A/ and MVSL-DSF_B/")
!python /content/DSE/unified_eval.py
